In [12]:
from typing import List, Tuple, Dict, Set
from collections import deque
import math
import numpy as np

## Tabu Search Implementaton

Minimise $\sum_{}^{j}T_j$ where $T_j = max(0, C_j-d_j)$

Solve a $1|prec|\sum_{}^{j}T_j$ problem with rejection method and penalty function method.

**Input:** DAG (in adjacency graph)

**Output:** Schedule (ordered list)

In [ ]:
def preprocessing(num_jobs: int, prec: List[Tuple[int, int]], p: List[int], d: List[int]
                  ) -> Tuple[Dict[int, Set[int]], Dict[int, Set[int]], Dict[int, Dict[str, int]]]:
    
    successors = {i: set() for i in range(1, num_jobs + 1)}
    predecessors = {i: set() for i in range(1, num_jobs + 1)}

    # Adjacency list
    for u, v in prec:
        successors[u].add(v)
        predecessors[v].add(u)

    # Processing time and due date
    job_info = {i: {'processing time':p[i-1], 'due date': d[i-1]} for i in range(1, num_jobs + 1)}

    return successors, predecessors, job_info



def initialise_schedule(num_jobs: int, pred: Dict[int, Set[int]], suc: Dict[int, Set[int]]
                        ) -> List[int]:
    ''' 
        BFS for initialising a valid schedule

        Inputs:
            num_jobs: number of jobs need to be scheduled
            pred: predecessor dictionary
            suc: successor dictionary

        Return:
            a valid schedule that respect the precedences.
    '''
    schedule = []

    in_degree = {i: len(pred[i]) for i in range(1, num_jobs + 1)}
    queue = deque([job for job, deg in in_degree.items() if deg == 0])

    while queue:
        curr = queue.popleft()
        schedule.append(curr)

        for s in suc[curr]:
            in_degree[s] -= 1
            if in_degree[s] == 0:
                queue.append(s)
    return schedule



def tardiness(x: List[int], job_info: Dict[int, Dict[str, int]]) -> int:
    completion_time = 0
    tardilist = []

    for job in x:
        completion_time += job_info[job]['processing time']
        tardilist.append(max(0, completion_time - job_info[job]['due date']))
    
    return sum(tardilist)



def check_precedence(x: List[int], prec: List[Tuple[int, int]], lmbda = int, job_info = Dict[int, Dict[str, int]], method: str = 'rejection'
                     ) -> Tuple[bool, float]:
    '''
        Check if precedence of a schedule violates the rules.

        Inputs:
            x: current job schedule
            prec: a list of all precedences (u, v) where u is the predecessor of v
            lmbda: penalty weight parameter
            job_info: containing processing times and due dates per job.
            method: choose either 'rejection' for rejection method or 'penalty' for penalty function method.

        Returns:
            (valid_precedence, total_penalty_cost)
    '''
    position = {job: i for i, job in enumerate(x)}
    penalty = 0
    is_valid = True

    for u, v in prec:
        if u in position and v in position:
            if position[u] > position[v]:
                is_valid = False
                penalty += lmbda * max(0, (position[u] - position[v])) # h_k measures the sum of magnitude of violations in the current schedule

    if method == 'rejection':
        return (is_valid, 0.0)

    elif method == 'penalty':
        cost = tardiness(x, job_info) + penalty
        return (is_valid, cost)

    else:
        raise ValueError("Invalid method specified. Options: 'rejection', 'penalty'.")



def tabu(p: List[int], d: List[int], prec: List[Tuple[int, int]], K: int, gamma: int, lmbda: int, L: int, x0 = None, method = 'rejection'
         ) -> List[int]:
    '''
        Compute the best schedule based on tabu search.

        Inputs:
            p: processing time
            d: due dates
            prec: a list of all precedences (u, v) where u is the predecessor of v
            K: maximum iterations
            gamma: threshold
            L: tabu list length
            handling: 0 for rejection method, 1 for penalty function method. Any other int is not valid.
            x0: initial schedule (if given)
            method: choose either 'rejection' for rejection method or 'penalty' for penalty function method.

        Returns:
            an optimised schedule as an ordered list.
    '''
    num_jobs = max(max(u, v) for u, v in prec) # detect num of jobs
    pred, suc, job_info = preprocessing(num_jobs, prec, p, d)
    if not x0:
        x0 = initialise_schedule(num_jobs, pred, suc)
    tabu_list = []
    t_best = math.inf
    x_best = x0 # best schedule

    for k in range(K+1):
        candidates = [] # all the y generated from the current schedule
        if k == 0: # iteration 0
            t_best = tardiness(x0, job_info)
        else: # all other iterations
            t = []
            swaps = []
            for i in range(num_jobs - 1):
                swap = (i, i+1)
                swaps.append(swap)

                if swap in tabu_list: # check legal swap
                    continue
                else:
                    x_tmp = x_best.copy()
                    x_tmp[i], x_tmp[i+1] = x_tmp[i+1], x_tmp[i] # swap
                    valid_schedule, cost = check_precedence(x_tmp, prec, lmbda, job_info, method) # check if swap was legal


                    if method == 'penalty':
                        c = cost # the cost in penalty function method is replaced by the cost from penalty function.
                    elif valid_schedule: # rejection method and valid schedule
                        c = tardiness(x_tmp, job_info)


                    if c <= t_best + gamma: ######### Not sure here!!!
                        if method == 'penalty' or valid_schedule:
                            t.append(c)
                            candidates.append(x_tmp)
            
            if not t:
                if min(t) < t_best:
                    t_best = min(t)
                    x_best = candidates[np.argmin(t)]
                    if len(tabu_list) > L:
                        tabu_list.pop(0)
                    tabu_list.append(swaps[np.argmin(t)])


    return x_best

### Test Cases

In [ ]:
prec = [(0, 1), (0, 2), (0, 4), (0, 5), (0, 6), (1, 7), (1, 8), (2, 3), (2, 9), 
         (3, 10), (4, 11), (5, 12), (6, 13), (7, 14), (8, 14), (9, 15), (10, 19), 
         (11, 16), (12, 17), (13, 20), (14, 18), (15, 19), (16, 26), (17, 20), 
         (18, 24), (19, 21), (20, 21), (20, 22), (21, 23), (21, 25), (22, 25), 
         (23, 27), (24, 26), (25, 26), (26, 28), (27, 29), (28, 29), (29, 30)
] # write in report the assumption for input

p = [2, 6, 14, 3, 3, 5, 2, 3, 2, 3, 2, 3, 6, 2, 10, 2, 2, 5, 18, 6, 14, 6, 2, 5, 5, 6, 5, 5, 10, 3, 10]
d = [329, 307, 340, 122, 73, 287, 61, 164, 256, 181, 88, 279, 290, 18, 217, 71, 307, 295, 209, 77, 217, 233, 71, 355, 141, 34, 93, 253, 82, 172, 269]